# 04. Baselines Avanzados — Día 4

**Objetivos**:
1. Entrenar **LightGBM** (4to baseline)
2. Analizar **feature importance** (top 20) en RF, XGB, LGBM — verificar no-leakage
3. Comparativa **CON vs SIN features `pliego_*`** — cuantificar aporte de la extracción Gemini
4. Decisión final: top 2 modelos para tuning Día 5

**Outputs**:
- `data/modelos/lgbm.pkl`
- `data/feature_importance.csv`
- `data/comparativa_pliego.csv`
- `figuras/dia_04_*.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

sns.set_style('whitegrid')
RAIZ_MODELO = Path('..').resolve()
DATA = RAIZ_MODELO / 'data'
FIGS = RAIZ_MODELO / 'figuras'
MODELOS = DATA / 'modelos'
RANDOM_STATE = 42
print('Imports OK')

In [ ]:
# Cargar splits y modelos previos
X_train = pd.read_parquet(DATA / 'X_train.parquet')
X_test = pd.read_parquet(DATA / 'X_test.parquet')
y_train = pd.read_parquet(DATA / 'y_train.parquet').iloc[:, 0]
y_test = pd.read_parquet(DATA / 'y_test.parquet').iloc[:, 0]

rf = joblib.load(MODELOS / 'rf.pkl')
xgbm = joblib.load(MODELOS / 'xgb.pkl')

print(f'Splits y modelos previos cargados')
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')

## 1. LightGBM

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos

lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=-1,           # sin limite (LGBM controla con num_leaves)
    num_leaves=63,
    learning_rate=0.1,
    objective='binary',
    metric='auc',
    scale_pos_weight=spw,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1,
)

t0 = time.time()
lgbm.fit(X_train, y_train)
lgbm_train_time = time.time() - t0

lgbm_proba = lgbm.predict_proba(X_test)[:, 1]
lgbm_pred = (lgbm_proba >= 0.5).astype(int)

print(f'LGBM entrenado en {lgbm_train_time:.1f}s')
print(f'  AUC: {roc_auc_score(y_test, lgbm_proba):.4f}')
print(f'  F1:  {f1_score(y_test, lgbm_pred):.4f}')
print(f'  P:   {precision_score(y_test, lgbm_pred):.4f}')
print(f'  R:   {recall_score(y_test, lgbm_pred):.4f}')

joblib.dump(lgbm, MODELOS / 'lgbm.pkl')

## 2. Feature Importance (top 20) — verificación de no-leakage

Si alguna feature "sospechosa" aparece arriba (ej. `dias_adicionados`),
habría leakage. Esperamos ver: modalidad de contratación, valor, entidad, departamento, etc.

In [ ]:
# Helper para extraer importance estandarizada
def importance_df(modelo, modelo_name, feature_names):
    if hasattr(modelo, 'feature_importances_'):
        imp = modelo.feature_importances_
    else:
        imp = np.abs(modelo.coef_[0])
    df = pd.DataFrame({
        'feature': feature_names,
        f'importance_{modelo_name}': imp,
    })
    df[f'rank_{modelo_name}'] = df[f'importance_{modelo_name}'].rank(ascending=False).astype(int)
    return df

fi_rf = importance_df(rf, 'RF', X_train.columns)
fi_xgb = importance_df(xgbm, 'XGB', X_train.columns)
fi_lgbm = importance_df(lgbm, 'LGBM', X_train.columns)

fi_full = fi_rf.merge(fi_xgb, on='feature').merge(fi_lgbm, on='feature')
# Rank promedio
fi_full['rank_promedio'] = fi_full[['rank_RF', 'rank_XGB', 'rank_LGBM']].mean(axis=1)
fi_full = fi_full.sort_values('rank_promedio')

print('Top 20 features (rank promedio entre RF, XGB, LGBM):')
print(fi_full.head(20)[['feature', 'rank_RF', 'rank_XGB', 'rank_LGBM', 'rank_promedio']].to_string(index=False))

fi_full.to_csv(DATA / 'feature_importance.csv', index=False)
print(f'\nGuardado: data/feature_importance.csv')

In [ ]:
# Plot top 20 por modelo
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (model_name, importance_col) in zip(axes, [
    ('RandomForest', 'importance_RF'),
    ('XGBoost', 'importance_XGB'),
    ('LightGBM', 'importance_LGBM'),
]):
    top = fi_full.nlargest(20, importance_col)
    top_sorted = top.sort_values(importance_col)
    ax.barh(top_sorted['feature'], top_sorted[importance_col], color='steelblue')
    ax.set_title(f'{model_name} — Top 20 features')
    ax.set_xlabel('importance')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(FIGS / 'dia_04_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Verificar que ninguna col sospechosa de leakage este entre top 30
COLS_SOSPECHOSAS = ['dias_adicionados', 'tiempo', 'presupuesto', 'alcance',
                     'n_modif_general', 'tiene_suspension', 'ratio_extension_duracion']

top30 = fi_full.head(30)['feature'].tolist()
presentes = [c for c in COLS_SOSPECHOSAS if c in top30]
if presentes:
    print(f'⚠️  ALERTA: posibles leaks en top 30: {presentes}')
else:
    print('✅ Ninguna col sospechosa de leakage en top 30 — pipeline limpio')

## 3. Comparativa: con vs sin features `pliego_*`

Reentrenamos XGBoost (el mejor) en dos versiones:
- **Versión A (con pliego)**: las 94 features actuales
- **Versión B (sin pliego)**: las 94 - 6 (eliminando los 6 indicadores) - 6 (eliminando sus flags `_was_nan`)

Si el delta de AUC es despreciable, los `pliego_*` no aportan en este momento
(aunque podrían aportar más con mayor cobertura de extracción).

In [ ]:
pliego_cols = [c for c in X_train.columns if c.startswith('pliego_')]
print(f'Cols pliego_*: {len(pliego_cols)} ({pliego_cols})')

# Quitar tambien los flags _was_nan que tengan que ver con pliego
pliego_flags = [c for c in X_train.columns
                if c.startswith('pliego_') and c.endswith('_was_nan')]
cols_a_drop = list(set(pliego_cols + pliego_flags))
print(f'Total cols a drop para version SIN pliego: {len(cols_a_drop)}')

X_train_no_pliego = X_train.drop(columns=cols_a_drop)
X_test_no_pliego = X_test.drop(columns=cols_a_drop)
print(f'  Sin pliego shape train: {X_train_no_pliego.shape}')

In [ ]:
# Entrenar XGB en ambas versiones (mismos params)
def train_xgb(Xtr, ytr, Xts, yts, label):
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        objective='binary:logistic', eval_metric='auc',
        scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_STATE,
        tree_method='hist',
    )
    m.fit(Xtr, ytr)
    proba = m.predict_proba(Xts)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'version': label,
        'n_features': Xtr.shape[1],
        'AUC': roc_auc_score(yts, proba),
        'F1': f1_score(yts, pred),
        'Precision': precision_score(yts, pred),
        'Recall': recall_score(yts, pred),
        'Accuracy': accuracy_score(yts, pred),
    }

res_con_pliego = train_xgb(X_train, y_train, X_test, y_test, 'CON pliego_*')
res_sin_pliego = train_xgb(X_train_no_pliego, y_train, X_test_no_pliego, y_test, 'SIN pliego_*')

comp = pd.DataFrame([res_con_pliego, res_sin_pliego]).round(4)
print(comp.to_string(index=False))

comp.to_csv(DATA / 'comparativa_pliego.csv', index=False)
delta_auc = res_con_pliego['AUC'] - res_sin_pliego['AUC']
print(f'\nDelta AUC = {delta_auc:+.4f} (positivo = pliego_* aporta)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
metricas = ['AUC', 'F1', 'Precision', 'Recall', 'Accuracy']
x = np.arange(len(metricas))
width = 0.4

vals_con = [res_con_pliego[m] for m in metricas]
vals_sin = [res_sin_pliego[m] for m in metricas]

ax.bar(x - width/2, vals_con, width, label='CON pliego_*', color='steelblue')
ax.bar(x + width/2, vals_sin, width, label='SIN pliego_*', color='lightcoral')

for i, (vc, vs) in enumerate(zip(vals_con, vals_sin)):
    ax.text(i - width/2, vc + 0.005, f'{vc:.3f}', ha='center', fontsize=8)
    ax.text(i + width/2, vs + 0.005, f'{vs:.3f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metricas)
ax.set_ylim(0, 1)
ax.set_title('Impacto de las features pliego_* (XGBoost)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / 'dia_04_impacto_pliego.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Resumen y shortlist para Día 5

In [ ]:
# Re-evaluacion completa con LGBM agregado
rf_proba = rf.predict_proba(X_test)[:, 1]
xgb_proba = xgbm.predict_proba(X_test)[:, 1]

rows = [
    {'modelo': 'RandomForest', 'AUC': roc_auc_score(y_test, rf_proba),
     'F1': f1_score(y_test, (rf_proba>=0.5).astype(int))},
    {'modelo': 'XGBoost', 'AUC': roc_auc_score(y_test, xgb_proba),
     'F1': f1_score(y_test, (xgb_proba>=0.5).astype(int))},
    {'modelo': 'LightGBM', 'AUC': roc_auc_score(y_test, lgbm_proba),
     'F1': f1_score(y_test, (lgbm_proba>=0.5).astype(int))},
]

resumen = pd.DataFrame(rows).sort_values('AUC', ascending=False).round(4)
print('Ranking final por AUC:')
print(resumen.to_string(index=False))

top2 = resumen.head(2)['modelo'].tolist()
print(f'\n🏆 Top 2 para tuning Día 5: {top2}')

## Conclusiones del Día 4

Ver `data/feature_importance.csv` y `data/comparativa_pliego.csv` para detalle.

**Decisiones cerradas:**
- 4 modelos baseline evaluados
- Feature importance valida no hay leakage residual
- Aporte de `pliego_*` cuantificado (con cobertura actual ~2-3%)
- Top 2 modelos seleccionados para tuning Día 5

**Próximo (Día 5)**: Random Search 80 iters × top 2 modelos × CV 5-fold.